In [ ]:
pip install pandas numpy

In [ ]:
import pandas as pd
from google.colab import userdata

data_path = userdata.get('DATA_PATH')
df = pd.read_csv(data_path)

print("=== Veri Seti Özeti ===")
row, column = df.shape
describe = df.describe()
cols = df.columns
print(f"Satır Sayısı = {row}, Sütun Sayısı = {column}")
print(f"Açıklama = {describe}")
print(f"Kolonlar = {cols}")

=== Veri Seti Özeti ===
Satır Sayısı = 69967, Sütun Sayısı = 39
Açıklama =           Unnamed: 0  Stok-Hizmet Kodu        Miktar    Birim Fiyat  \
count   69967.000000      69967.000000  69967.000000   69967.000000   
mean    53786.652851       2019.516286      3.986379     125.868109   
std     31087.556954       1769.870522      8.332894     994.806151   
min         1.000000          1.000000      1.000000       0.000000   
25%     27118.500000        692.000000      1.000000       9.000000   
50%     54378.000000       1292.000000      2.000000      24.000000   
75%     80579.500000       3291.000000      4.000000      60.000000   
max    107475.000000       6852.000000    500.000000  200000.000000   

               Tutar           Kur  İndirim %  İndirim Tutarı         KDV %  \
count   69967.000000  69967.000000    69967.0         69967.0  69967.000000   
mean     2434.851269     28.148376        0.0             0.0     18.243315   
std      4756.725249     13.475222        0.0   

/tmp/ipykernel_537/3771570212.py:5: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


In [ ]:
df = df[['Stok-Hizmet Kodu', 'Tarih_Converted', 'Miktar']]
df = df.rename(columns={'Stok-Hizmet Kodu':'product', 'Tarih_Converted':'date', 'Miktar':'quantity'})

df.head()

,product,date,quantity
0,1349,2025-12-13,1.0
1,1348,2025-12-13,1.0
2,1347,2025-12-13,1.0
3,1346,2025-12-13,1.0
4,5312,2025-12-13,1.0


In [ ]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['product', 'date'])

df['week'] = df['date'].dt.to_period('W-SUN').dt.start_time

df = df.groupby(['product', 'week'])['quantity'].sum().reset_index().copy()

all_products = df['product'].unique()
full_range = pd.date_range(start=df['week'].min(), end=df['week'].max(), freq='W-MON')

In [ ]:
full_index = pd.MultiIndex.from_product(
    [all_products, full_range],
    names=['product', 'week']
)

df = (
    df.set_index(['product', 'week'])
      .reindex(full_index, fill_value=0)
      .reset_index().copy()
)

In [ ]:
print(" === Düzenleme Sonrası === ")
print(f"Satır ve Sütun Sayısı {df.shape}")
print(f"df = {df.head}")

 === Düzenleme Sonrası === 
Satır ve Sütun Sayısı (893834, 3)
df = <bound method NDFrame.head of         product       week  quantity
0             1 2022-01-03       0.0
1             1 2022-01-10       0.0
2             1 2022-01-17       0.0
3             1 2022-01-24       0.0
4             1 2022-01-31       0.0
...         ...        ...       ...
893829     6852 2025-11-10       0.0
893830     6852 2025-11-17       0.0
893831     6852 2025-11-24       0.0
893832     6852 2025-12-01       0.0
893833     6852 2025-12-08       2.0

[893834 rows x 3 columns]>
